# Tool-Router Ladder: router load test (V1-18)

Routing latency per decision at concurrency 1, 8 and 32 (p50 / p95 / p99), for the 1.5B
student at catalog 80: the vLLM server with log-probs on, the router service in front of it,
and 256 replayed prompts per level. About 30 minutes on a T4.

**No API quota:** the router runs with `--no-teacher`, so an escalation is recorded, never sent.
This can run on any day, including a day the Mac harvest is running.

**Settings before committing:** Accelerator GPU T4, Internet on, the tool-router dataset
attached (it holds `prompts-c80.jsonl`), secret `HF_TOKEN` ticked.
Then Save Version → **Save & Run All (Commit)**.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
!pip install -q vllm

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# The router service needs LangGraph (Stage D's notebook does not).
!pip install -q "langgraph>=0.2"


In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# The student: serving.server_command, as Stage D starts it (log-probs on).
import subprocess, sys
sys.path.insert(0, ".")
from src.config import load_config
from src.generate import _resolve_adapter
from src.serving import server_command, wait_ready

cfg = load_config("configs/qwen15b.yaml")
cmd = server_command(cfg, _resolve_adapter(f"{HF_USER}/tool-router-qwen15b-sft"), 32768, 8765, logprobs=True)
student = subprocess.Popen(cmd, stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT)
wait_ready("http://127.0.0.1:8765", alive=lambda: student.poll() is None)
print("student ready")

In [ ]:
# The router, without a teacher.
router = subprocess.Popen(
    ["python", "src/router_service.py", "--config", "configs/qwen15b.yaml", "--catalog", "80",
     "--threshold", "0.7", "--student-url", "http://127.0.0.1:8765", "--port", "8080", "--no-teacher"],
    stdout=open("/kaggle/working/router.log", "w"), stderr=subprocess.STDOUT,
)
wait_ready("http://127.0.0.1:8080", timeout_s=300, alive=lambda: router.poll() is None)
print("router ready")

In [ ]:
!python src/load_test.py --url http://127.0.0.1:8080 --prompts "$DATA/prompts-c80.jsonl" --label qwen15b-sft-c80 --out "$OUT/load-test-qwen15b-sft-c80.json"

In [ ]:
from huggingface_hub import HfApi

router.terminate(); student.terminate()
REPO = f"{HF_USER}/tool-router-results"
api = HfApi()
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, path_in_repo="results", repo_id=REPO, repo_type="dataset")
print("uploaded:", sorted(os.listdir(OUT)))